# Approach 1: Context Window / In-Context Learning with Tabular Data

The simplest approach: **serialize a table as text and inject it into the LLM's prompt**.

**How it works:**
- Convert DataFrame rows to a markdown or CSV string
- Prepend to the question as context
- The LLM answers based on what it sees in the prompt

**Strengths:** Zero setup, works with any LLM, no training needed  
**Weaknesses:** Context size limits, no learned relationships, expensive at scale

In [ ]:
# Install if needed
# !pip install transformers torch pandas

In [ ]:
import pandas as pd
from transformers import pipeline

# Load tabular data
df = pd.read_csv('../data/employees.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## Step 1: Serialize the table

We convert the DataFrame to a markdown table string that the LLM can read.

In [ ]:
def serialize_table_markdown(df):
    """Convert a DataFrame to a markdown table string."""
    return df.to_markdown(index=False)

def serialize_table_csv(df):
    """Convert a DataFrame to a CSV string."""
    return df.to_csv(index=False)

def serialize_table_sentences(df):
    """Convert each row to a natural language sentence."""
    sentences = []
    for _, row in df.iterrows():
        s = (f"{row['name']} works in {row['department']} with a salary of "
             f"${row['salary']:,}, {row['years_experience']} years of experience, "
             f"a performance score of {row['performance_score']}, and has "
             f"{'been' if row['promoted'] == 'yes' else 'not been'} promoted.")
        sentences.append(s)
    return ' '.join(sentences)

# Show each format
print("=== Markdown Format ===")
print(serialize_table_markdown(df.head(5)))
print()
print("=== CSV Format ===")
print(serialize_table_csv(df.head(3)))
print()
print("=== Sentences Format ===")
print(serialize_table_sentences(df.head(3)))

## Step 2: Build a prompt with the table as context

In [ ]:
def build_prompt(df, question):
    table_str = serialize_table_markdown(df)
    prompt = f"""Here is a table of employee data:

{table_str}

Question: {question}
Answer:"""
    return prompt

questions = [
    "Who has the highest salary?",
    "How many employees are in Engineering?",
    "Which department has the best average performance score?",
]

for q in questions:
    print(f"Q: {q}")
    # Compute programmatically for reference
    if "highest salary" in q:
        ans = df.loc[df['salary'].idxmax(), 'name']
    elif "Engineering" in q:
        ans = len(df[df['department'] == 'Engineering'])
    else:
        ans = df.groupby('department')['performance_score'].mean().idxmax()
    print(f"True answer (computed): {ans}\n")

## Step 3: Use GPT-2 as the LLM

> **Note:** GPT-2 is a base model (not instruction-tuned), so its answers will be imperfect. 
> In production you'd use an instruction-tuned model (GPT-4, Llama-3-Instruct, etc.).
> The goal here is to show the **mechanism**, not get perfect answers.

In [ ]:
print("Loading GPT-2...")
generator = pipeline('text-generation', model='gpt2', device=-1)
print("Model loaded!")

In [ ]:
question = "Who has the highest salary?"
prompt = build_prompt(df, question)

print(f"Prompt length: {len(prompt)} characters")
print(f"\nQuestion: {question}")
print("-" * 50)

output = generator(
    prompt,
    max_new_tokens=30,
    num_return_sequences=1,
    do_sample=False,  # greedy for determinism
    pad_token_id=50256
)

generated = output[0]['generated_text']
# Extract only the answer part (after "Answer:")
answer = generated.split("Answer:")[-1].strip()
print(f"LLM Answer: {answer}")
print(f"True Answer: {df.loc[df['salary'].idxmax(), 'name']}")

## Step 4: Context window limits — what happens with large tables?

In [ ]:
# GPT-2 has a 1024 token context window
# Let's see how fast a table fills it

from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Simulate a larger dataset
large_df = pd.concat([df] * 10, ignore_index=True)  # 200 rows

for n_rows in [5, 10, 20, 50, 100, 200]:
    prompt = build_prompt(df.head(n_rows), "Who earns more than $90,000?")
    tokens = tokenizer.encode(prompt)
    fits = len(tokens) <= 1024
    print(f"{n_rows:3d} rows → {len(tokens):5d} tokens  {'✓ fits in GPT-2 context' if fits else '✗ exceeds context window'}")

## Key Takeaways

| | Context Window Approach |
|---|---|
| **Setup** | None — any LLM works |
| **Scale** | Limited by context window (~1K–200K tokens) |
| **Accuracy** | Depends on model quality; base models struggle |
| **Cost** | Tokens per query × table size |
| **Best for** | Small tables, quick prototypes, one-off queries |

**Next:** See `02_text_to_sql.ipynb` where the LLM generates SQL instead of reading the raw data.